In [0]:
STORAGE_ACCOUNT = "stlakehousedev225"
CATALOG          = "ecommerce_dev"
BRONZE_SCHEMA    = "bronze"
SILVER_SCHEMA    = "silver"
SILVER_BASE      = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

In [0]:
# SET UNITY CATALOG CONTEXT

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SILVER_SCHEMA}")
print(f"Using catalog: {CATALOG}.{SILVER_SCHEMA}")

In [0]:
from delta.tables import DeltaTable

# Read from bronze
df_orders = spark.table("ecommerce_dev.bronze.orders")

# Check if table exists
if spark.catalog.tableExists("ecommerce_dev.silver.orders"):
    target = DeltaTable.forName(spark, "ecommerce_dev.silver.orders")

    target.alias("target").merge(
        df_orders.alias("source"),
        "target.order_id = source.order_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

    print(f"silver.orders MERGED — {spark.table('ecommerce_dev.silver.orders').count()} rows")

else:
    df_orders.write \
        .format("delta") \
        .mode("overwrite") \
        .option("path", f"{SILVER_BASE}orders") \
        .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.orders")

    print(f"silver.orders CREATED — {df_orders.count()} rows")

In [0]:
from pyspark.sql.functions import col, expr

df_reviews = spark.table("ecommerce_dev.bronze.reviews")

df_reviews = df_reviews \
    .filter(col("review_score").isin("1", "2", "3", "4", "5")) \
    .withColumn("review_score", col("review_score").cast("integer")) \
    .withColumn("review_creation_date", expr("try_to_timestamp(review_creation_date)")) \
    .withColumn("review_answer_timestamp", expr("try_to_timestamp(review_answer_timestamp)"))

df_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", f"{SILVER_BASE}reviews") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.reviews")

print(f"silver.reviews done — {df_reviews.count()} rows")

In [0]:
from pyspark.sql.functions import col, to_timestamp

df_order_items = spark.table("ecommerce_dev.bronze.order_items")

df_order_items = df_order_items \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double")) \
    .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date"), "yyyy-MM-dd HH:mm:ss"))

df_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", f"{SILVER_BASE}order_items") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.order_items")

print(f"silver.order_items done — {df_order_items.count()} rows")

In [0]:
# Read
df_customers = spark.table("ecommerce_dev.bronze.customers")

# Write
df_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", f"{SILVER_BASE}customers") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.customers")

print(f"silver.customers done — {df_customers.count()} rows")

In [0]:
df_products = spark.table("ecommerce_dev.bronze.products")

df_products = df_products \
    .withColumn("product_weight_g", col("product_weight_g").cast("double")) \
    .withColumn("product_length_cm", col("product_length_cm").cast("double")) \
    .withColumn("product_height_cm", col("product_height_cm").cast("double")) \
    .withColumn("product_width_cm", col("product_width_cm").cast("double"))

df_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", f"{SILVER_BASE}products") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.products")

print(f"silver.products done — {df_products.count()} rows")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# Read from bronze
df_payments = spark.table("ecommerce_dev.bronze.payments")

df_payments = df_payments \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .withColumn("payment_installments", col("payment_installments").cast("integer"))

# Check if silver.payments already exists
if spark.catalog.tableExists("ecommerce_dev.silver.payments"):
    # MERGE — upsert into existing table
    target = DeltaTable.forName(spark, "ecommerce_dev.silver.payments")

    target.alias("target").merge(
        df_payments.alias("source"),
        "target.order_id = source.order_id AND target.payment_sequential = source.payment_sequential"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

    print(f"silver.payments MERGED — {spark.table('ecommerce_dev.silver.payments').count()} rows")

else:
    # First run — just write
    df_payments.write \
        .format("delta") \
        .mode("overwrite") \
        .option("path", f"{SILVER_BASE}payments") \
        .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.payments")

    print(f"silver.payments CREATED — {df_payments.count()} rows")

In [0]:
df_sellers = spark.table("ecommerce_dev.bronze.sellers")

df_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", f"{SILVER_BASE}sellers") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.sellers")

print(f"silver.sellers done — {df_sellers.count()} rows")

In [0]:
silver_tables = ["orders", "reviews", "customers", "products", "payments", "order_items", "sellers"]

print(f"{'Table':<20} {'Rows':>10}")
print("-" * 32)

for table in silver_tables:
    count = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SILVER_SCHEMA}.{table}").collect()[0][0]
    print(f"{table:<20} {count:>10,}")

In [0]:
def add_constraint_if_not_exists(table, name, rule):
    try:
        spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} CHECK ({rule})")
        print(f"Added: {name}")
    except Exception as e:
        if "already exists" in str(e):
            print(f"Already exists: {name}")
        else:
            raise e

add_constraint_if_not_exists("ecommerce_dev.silver.orders", "orders_id_not_null", "order_id IS NOT NULL")
add_constraint_if_not_exists("ecommerce_dev.silver.payments", "positive_payment_value", "payment_value >= 0")
add_constraint_if_not_exists("ecommerce_dev.silver.reviews", "valid_review_score", "review_score BETWEEN 1 AND 5")
add_constraint_if_not_exists("ecommerce_dev.silver.order_items", "positive_price", "price > 0")

print("All constraints done.")